# Ví dụ đơn giản: Overfitting trong hồi quy giá nhà

Ví dụ chỉ dùng **diện tích nhà** để dự báo **giá nhà**. Mục tiêu là quan sát rõ:

1. Hồi quy tuyến tính đơn giản học quy luật tổng quát tốt.
2. Hồi quy đa thức bậc cao khớp cả nhiễu và bị overfitting.
3. Giảm độ phức tạp, Ridge và cross-validation giúp hạn chế overfitting.

Đơn vị giá là **tỷ đồng**. Dữ liệu được mô phỏng để phục vụ học tập.

## 1. Import thư viện

Nếu máy chưa có thư viện:

```bash
pip install numpy pandas matplotlib scikit-learn
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Tương thích với cả Matplotlib cũ và mới.
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

pd.set_option("display.float_format", lambda value: f"{value:.4f}")

## 2. Tạo dữ liệu giá nhà

Ta giả sử quan hệ thật giữa diện tích $x$ và giá nhà $y$ là:

$$y = 1 + 0.06x + \varepsilon,$$

trong đó:

- $x$ là diện tích nhà, từ khoảng 35 đến 200 m²;
- $y$ là giá nhà, đơn vị tỷ đồng;
- $\varepsilon \sim \mathcal{N}(0, 0.7^2)$ là nhiễu do vị trí, chất lượng xây dựng và các yếu tố chưa quan sát.

Mô hình không nên cố gắng khớp hoàn toàn nhiễu $\varepsilon$ vì nhiễu không tạo thành quy luật tổng quát.

In [ ]:
RANDOM_SEED = 1
SPLIT_SEED = 37
N_SAMPLES = 70

rng = np.random.default_rng(RANDOM_SEED)

area = np.sort(rng.uniform(35, 200, N_SAMPLES))
true_price = 1 + 0.06 * area
observed_price = true_price + rng.normal(0, 0.7, N_SAMPLES)

df = pd.DataFrame({
    "area_m2": area,
    "price_billion_vnd": observed_price,
    "true_price": true_price,
})

df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(
    df["area_m2"],
    df["price_billion_vnd"],
    color="#2563EB",
    alpha=0.75,
    s=48,
    label="Dữ liệu quan sát",
)
ax.plot(
    df["area_m2"],
    df["true_price"],
    "k--",
    linewidth=2.2,
    label="Quan hệ thật",
)
ax.set(
    title="Dữ liệu giá nhà có nhiễu",
    xlabel="Diện tích nhà (m²)",
    ylabel="Giá nhà (tỷ đồng)",
)
ax.legend()
plt.show()

## 3. Chia Train, Validation và Test

- **Train:** dùng để ước lượng các hệ số của mô hình.
- **Validation:** dùng để chọn bậc đa thức hoặc mức regularization.
- **Test:** chỉ dùng sau khi đã chốt mô hình, nhằm đánh giá khả năng dự báo dữ liệu mới.

Tỷ lệ chia là 60% Train, 20% Validation và 20% Test.

In [ ]:
X = df[["area_m2"]].to_numpy()
y = df["price_billion_vnd"].to_numpy()
all_idx = np.arange(N_SAMPLES)

train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.40, random_state=SPLIT_SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SPLIT_SEED
)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

train_val_idx = np.concatenate([train_idx, val_idx])
X_train_val, y_train_val = X[train_val_idx], y[train_val_idx]

print(f"Train:      {len(train_idx)} mẫu")
print(f"Validation: {len(val_idx)} mẫu")
print(f"Test:       {len(test_idx)} mẫu")

In [ ]:
colors = {
    "Train": "#2563EB",
    "Validation": "#F59E0B",
    "Test": "#DC2626",
}

fig, ax = plt.subplots(figsize=(10, 5.5))
for name, indices in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx),
]:
    ax.scatter(
        X[indices, 0],
        y[indices],
        s=50,
        alpha=0.78,
        color=colors[name],
        label=name,
    )

ax.plot(area, true_price, "k--", linewidth=2, label="Quan hệ thật")
ax.set(
    title="Phân chia dữ liệu",
    xlabel="Diện tích nhà (m²)",
    ylabel="Giá nhà (tỷ đồng)",
)
ax.legend(ncol=4)
plt.show()

## 4. Các hàm hỗ trợ

`PolynomialFeatures(degree=d)` biến một đầu vào $x$ thành:

$$[x, x^2, x^3, \ldots, x^d].$$

Sau phép biến đổi, `LinearRegression` vẫn là mô hình tuyến tính theo các hệ số:

$$\hat{y}=w_0+w_1x+w_2x^2+\cdots+w_dx^d.$$

Vì vậy, hồi quy đa thức là **hồi quy tuyến tính trên các đặc trưng đã biến đổi**.

In [ ]:
def make_polynomial_model(degree, estimator):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", estimator),
    ])


def regression_metrics(model, X_part, y_part):
    prediction = model.predict(X_part)
    return {
        "RMSE": mean_squared_error(y_part, prediction) ** 0.5,
        "MAE": mean_absolute_error(y_part, prediction),
        "R2": r2_score(y_part, prediction),
    }


def evaluate_three_sets(model):
    return pd.DataFrame({
        "Train": regression_metrics(model, X_train, y_train),
        "Validation": regression_metrics(model, X_val, y_val),
        "Test": regression_metrics(model, X_test, y_test),
    }).T


x_grid = np.linspace(30, 205, 700).reshape(-1, 1)
y_grid_true = 1 + 0.06 * x_grid.ravel()

## 5. Hồi quy tuyến tính đơn giản

Với `degree=1`, mô hình chỉ sử dụng đặc trưng $x$ và học một đường thẳng:

$$\hat{y}=w_0+w_1x.$$

Đây là mô hình phù hợp khi giá nhà tăng gần tuyến tính theo diện tích.

In [ ]:
linear_model_train = make_polynomial_model(1, LinearRegression())
linear_model_train.fit(X_train, y_train)

linear_metrics = evaluate_three_sets(linear_model_train)
linear_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(X_train[:, 0], y_train, color=colors["Train"], alpha=0.75, label="Train")
ax.scatter(X_val[:, 0], y_val, color=colors["Validation"], alpha=0.75, label="Validation")
ax.scatter(X_test[:, 0], y_test, color=colors["Test"], alpha=0.75, label="Test")
ax.plot(x_grid, y_grid_true, "k--", linewidth=2, label="Quan hệ thật")
ax.plot(
    x_grid,
    linear_model_train.predict(x_grid),
    color="#059669",
    linewidth=2.5,
    label="Hồi quy tuyến tính bậc 1",
)
ax.set(
    title="Mô hình bậc 1 học xu hướng tổng quát",
    xlabel="Diện tích nhà (m²)",
    ylabel="Giá nhà (tỷ đồng)",
)
ax.legend(ncol=3)
plt.show()

## 6. Tạo overfitting bằng hồi quy đa thức bậc 15

Mô hình bậc 15 có 15 đặc trưng $x, x^2, \ldots, x^{15}$ nhưng chỉ có 42 mẫu Train. Mô hình có đủ độ linh hoạt để uốn cong theo các dao động ngẫu nhiên của dữ liệu.

Ta kỳ vọng:

- Train RMSE giảm;
- Validation/Test RMSE tăng;
- đường dự báo dao động mạnh, đặc biệt ở vùng có ít điểm Train.

In [ ]:
overfit_model = make_polynomial_model(15, LinearRegression())
overfit_model.fit(X_train, y_train)

overfit_metrics = evaluate_three_sets(overfit_model)
overfit_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)

for ax, title, model in [
    (axes[0], "Hồi quy tuyến tính bậc 1", linear_model_train),
    (axes[1], "Hồi quy đa thức bậc 15", overfit_model),
]:
    ax.scatter(X_train[:, 0], y_train, color=colors["Train"], alpha=0.72, label="Train")
    ax.scatter(X_val[:, 0], y_val, color=colors["Validation"], alpha=0.72, label="Validation")
    ax.scatter(X_test[:, 0], y_test, color=colors["Test"], alpha=0.72, label="Test")
    ax.plot(x_grid, y_grid_true, "k--", linewidth=1.8, label="Quan hệ thật")
    ax.plot(x_grid, model.predict(x_grid), color="#7C3AED", linewidth=2.3, label="Dự báo")
    ax.set(title=title, xlabel="Diện tích (m²)", ylabel="Giá (tỷ đồng)")
    ax.legend()

axes[1].set_ylim(0, 15)
fig.suptitle("Mô hình bậc cao khớp cả nhiễu của dữ liệu", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

### Nhận biết overfitting

So sánh hai bảng chỉ số:

- Mô hình bậc 15 có Train RMSE thấp hơn mô hình bậc 1.
- Tuy nhiên, Validation RMSE và Test RMSE cao hơn rõ rệt.
- Khoảng cách lớn giữa lỗi Train và lỗi Validation/Test gọi là **generalization gap**.

Không nên chỉ nhìn vào độ chính xác trên Train để kết luận mô hình tốt.

## 7. Trực quan hóa độ phức tạp của mô hình

Ta thay đổi bậc đa thức từ 1 đến 15. Nếu bậc quá cao, lỗi Train thường tiếp tục giảm nhưng lỗi Validation tăng.

In [ ]:
degrees = list(range(1, 16))
train_rmse_by_degree = []
val_rmse_by_degree = []

for degree in degrees:
    model = make_polynomial_model(degree, LinearRegression())
    model.fit(X_train, y_train)
    train_rmse_by_degree.append(
        mean_squared_error(y_train, model.predict(X_train)) ** 0.5
    )
    val_rmse_by_degree.append(
        mean_squared_error(y_val, model.predict(X_val)) ** 0.5
    )

best_degree = degrees[int(np.argmin(val_rmse_by_degree))]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(degrees, train_rmse_by_degree, marker="o", color="#2563EB", label="Train RMSE")
ax.plot(degrees, val_rmse_by_degree, marker="o", color="#F59E0B", label="Validation RMSE")
ax.axvline(
    best_degree,
    color="#059669",
    linestyle="--",
    label=f"Bậc tốt nhất theo Validation = {best_degree}",
)
ax.set_xticks(degrees)
ax.set_xticklabels([str(degree) for degree in degrees])
ax.set(
    title="Độ phức tạp tăng làm xuất hiện overfitting",
    xlabel="Bậc đa thức",
    ylabel="RMSE (tỷ đồng)",
)
ax.legend()
plt.show()

## 8. Khắc phục 1 — Giảm độ phức tạp

Biện pháp đơn giản nhất là chọn mô hình có bậc thấp dựa trên Validation. Trong bộ dữ liệu này, Validation chọn `degree=1`.

Sau khi chọn bậc, ta huấn luyện lại mô hình bằng Train + Validation rồi mới đánh giá trên Test.

In [ ]:
simple_model = make_polynomial_model(best_degree, LinearRegression())
simple_model.fit(X_train_val, y_train_val)

simple_test_metrics = regression_metrics(simple_model, X_test, y_test)
pd.DataFrame(simple_test_metrics, index=[f"Degree = {best_degree}"])

## 9. Khắc phục 2 — Ridge regularization

Ta vẫn giữ đa thức bậc 15 nhưng thêm phần phạt L2:

$$\text{Loss}=\sum_i(y_i-\hat{y}_i)^2+\alpha\sum_jw_j^2.$$

- Nếu $\alpha=0$, Ridge trở thành hồi quy tuyến tính thông thường.
- Khi $\alpha$ tăng, các hệ số lớn bị phạt mạnh hơn và đường dự báo bớt dao động.
- Nếu $\alpha$ quá lớn, mô hình lại có thể underfit.

Ta dùng 5-fold cross-validation để chọn $\alpha$.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_search = GridSearchCV(
    estimator=make_polynomial_model(15, Ridge()),
    param_grid={"model__alpha": np.logspace(-6, 4, 30)},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)
ridge_search.fit(X_train_val, y_train_val)
ridge_model = ridge_search.best_estimator_

print(f"Alpha tốt nhất: {ridge_search.best_params_['model__alpha']:.6g}")
print(f"CV RMSE: {-ridge_search.best_score_:.4f} tỷ đồng")
pd.DataFrame(
    regression_metrics(ridge_model, X_test, y_test),
    index=["Degree 15 + Ridge"],
)

## 10. Khắc phục 3 — Chọn cả bậc đa thức và alpha bằng cross-validation

Thay vì chọn bậc bằng một lần chia Validation, ta có thể tìm đồng thời:

- `degree`: độ phức tạp của mô hình;
- `alpha`: mức regularization.

Cross-validation lặp lại việc huấn luyện trên nhiều phần dữ liệu, vì vậy kết quả ít phụ thuộc vào một lần chia duy nhất.

In [ ]:
tuned_search = GridSearchCV(
    estimator=make_polynomial_model(1, Ridge()),
    param_grid={
        "poly__degree": list(range(1, 11)),
        "model__alpha": np.logspace(-5, 3, 20),
    },
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)
tuned_search.fit(X_train_val, y_train_val)
tuned_model = tuned_search.best_estimator_

print("Siêu tham số tốt nhất:", tuned_search.best_params_)
print(f"CV RMSE: {-tuned_search.best_score_:.4f} tỷ đồng")
pd.DataFrame(
    regression_metrics(tuned_model, X_test, y_test),
    index=["Degree + alpha chọn bằng CV"],
)

## 11. So sánh kết quả cuối cùng

In [ ]:
models = {
    "Bậc 15 không regularization": overfit_model,
    f"Giảm độ phức tạp: bậc {best_degree}": simple_model,
    "Bậc 15 + Ridge": ridge_model,
    "Degree + alpha bằng CV": tuned_model,
}

comparison_rows = []
for name, model in models.items():
    prediction = model.predict(X_test)
    comparison_rows.append({
        "Mô hình": name,
        "Test RMSE": mean_squared_error(y_test, prediction) ** 0.5,
        "Test MAE": mean_absolute_error(y_test, prediction),
        "Test R2": r2_score(y_test, prediction),
    })

comparison = pd.DataFrame(comparison_rows).sort_values("Test RMSE")
comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, (name, model) in zip(axes.ravel(), models.items()):
    ax.scatter(X_train_val[:, 0], y_train_val, color="#2563EB", alpha=0.60, label="Train + Validation")
    ax.scatter(X_test[:, 0], y_test, color="#DC2626", alpha=0.82, label="Test")
    ax.plot(x_grid, y_grid_true, "k--", linewidth=1.8, label="Quan hệ thật")
    ax.plot(x_grid, model.predict(x_grid), color="#7C3AED", linewidth=2.3, label="Dự báo")
    test_rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5
    ax.set_title(f"{name}\nTest RMSE = {test_rmse:.3f}")
    ax.set_xlabel("Diện tích (m²)")
    ax.set_ylabel("Giá (tỷ đồng)")
    ax.set_ylim(0, 15)
    ax.legend()

fig.suptitle("So sánh trước và sau khi hạn chế overfitting", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

## 12. Kết luận

- Hồi quy tuyến tính bậc 1 học đúng xu hướng giá tăng theo diện tích.
- Hồi quy đa thức bậc 15 vẫn là mô hình tuyến tính theo tham số, nhưng có quá nhiều đặc trưng nên khớp cả nhiễu.
- Overfitting được nhận biết qua Train RMSE thấp trong khi Validation/Test RMSE cao.
- Giảm bậc đa thức là biện pháp đơn giản và dễ giải thích nhất.
- Ridge giữ được mô hình bậc cao nhưng hạn chế các hệ số lớn.
- Cross-validation giúp chọn `degree` và `alpha` có hệ thống hơn.
- Tập Test chỉ dùng để đánh giá cuối cùng, không dùng để lựa chọn mô hình.

Trong bài toán thực tế, có thể bổ sung các đặc trưng như vị trí, số phòng ngủ và tuổi nhà. Khi số đặc trưng tăng, cần tiếp tục kiểm tra data leakage, multicollinearity và độ ổn định của mô hình.